In [0]:
# Definición del destino analítico para la dimensión temporal fija
tabla_destino = "platform_dev.gold_byma.dim_fecha"

# Configuración de parámetros homogéneos para mantener simetría con el orquestador general
dbutils.widgets.text("full_load", "0")
full_load = int(dbutils.widgets.get("full_load"))
carga_full = (full_load == 1)

import logging
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("dimension_fecha")

In [0]:
try:
    # Programación Defensiva: Si la infraestructura analítica fue limpiada, el script
    # detecta la ausencia de la tabla y ejecuta el DDL maestro de creación.
    if not spark.catalog.tableExists(tabla_destino):
        logger.warning(f"La tabla {tabla_destino} no existe. Creando...")
        dbutils.notebook.run("../DDL/creacion_tablas", 0)
        carga_full = True # Forzamos la carga masiva inicial para el calendario
    else:
        # Si la tabla física existe pero no contiene registros, forzamos la carga
        # para garantizar que la tabla de hechos cuente con el mapa de joins disponible.
        tiene_datos = spark.sql(f"SELECT 1 FROM {tabla_destino} LIMIT 1").take(1) != []
        if not tiene_datos:
            carga_full = True
except Exception:
    logger.exception(f"Error al validar {tabla_destino}")
    raise

In [0]:
logger.info("Generando serie temporal del calendario para el período 2026")

# Generación controlada de filas: Creamos una serie matemática en memoria para cubrir
# la ventana temporal de datos operativos solicitada por el negocio.
df_calendario = (
    spark.range(0, 120)
    .withColumn("fecha", F.expr("DATE_ADD('2026-01-01', cast(id as int))"))
    # CONCEPTO CLAVE DE ARQUITECTURA: Usamos una 'Smart Key' (Clave Inteligente) entera (yyyyMMdd).
    # Al ser determinística e inmutable, no requiere una secuencia IDENTITY del motor ACID. 
    # Permite a la tabla de hechos particionar de forma nativa por ID entero sin costo de performance.
    .withColumn("fecha_sk", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("dia_semana", F.date_format("fecha", "EEEE"))
    # Atributos Analíticos Avanzados: Precalculamos banderas lógicas para optimizar las consultas
    # de los analistas de negocio en PowerBI (evita calcular funciones de fecha al vuelo en los reportes).
    .withColumn("es_fin_de_semana", F.dayofweek("fecha").isin(1, 7))
    .withColumn("es_feriado_bursatil", F.lit(False)) # Placeholder para futura integración con API de feriados de BYMA
    .select("fecha_sk", "fecha", "anio", "mes", "trimestre", "dia_semana", "es_fin_de_semana", "es_feriado_bursatil")
)

# Exposición en el catálogo de memoria temporal para la evaluación del motor Spark
df_calendario.createOrReplaceTempView("v_calendario_preparado")

In [0]:
try:
    # JUSTIFICACIÓN DE INGENIERÍA: A diferencia de Clientes, Canales o Instrumentos, acá el uso
    # de '.mode("overwrite")' es 100% SEGURO. Como las llaves 'fecha_sk' son fijas y basadas en el calendario, 
    # una sobreescritura jamás va a alterar o desfasar los IDs históricos guardados en Hechos.
    if carga_full or spark.table(tabla_destino).count() == 0:
        logger.warning(f"Escribiendo calendario completo en {tabla_destino}")
        df_calendario.write.format("delta").mode("overwrite").saveAsTable(tabla_destino)
        logger.info("Calendario guardado con éxito.")
    else:
        # Optimización de Recursos (FinOps): Si el proceso corre en modo incremental diario y la tabla
        # ya está poblada, omitimos la recarga completa para ahorrar tiempo de cómputo en el clúster.
        logger.info("La dimensión de fechas ya se encuentra poblada. Omitiendo recarga por optimización.")
except Exception:
    logger.exception(f"Error al escribir en {tabla_destino}")
    raise